# Lead Scoring — Post-Ingestion EDA

Data quality validation for the `leads` table after ingestion.

In [ ]:
import sys
import os

# Ensure project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sqlalchemy import create_engine, text

from config.settings import get_settings

%matplotlib inline
sns.set_theme(style='whitegrid')
matplotlib.rcParams['figure.dpi'] = 100

settings = get_settings()
engine = create_engine(settings.database.sync_url)

def query(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print('Engine ready:', engine.url)

## 1. Row Counts

In [ ]:
total = query('SELECT COUNT(*) AS total_leads FROM leads')
print('Total leads:', total['total_leads'].iloc[0])

by_source = query("""
    SELECT source_system, COUNT(*) AS count
    FROM leads
    GROUP BY source_system
    ORDER BY count DESC
""")
display(by_source)

## 2. Null Analysis

In [ ]:
columns = [
    'id', 'external_id', 'source_system', 'lead_origin', 'lead_source',
    'country', 'city', 'current_occupation', 'specialization',
    'do_not_email', 'do_not_call', 'total_visits', 'total_time_spent',
    'page_views_per_visit', 'last_activity', 'tags', 'converted',
    'created_at', 'updated_at'
]

null_exprs = ', '.join(
    f"ROUND(100.0 * SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS {c}"
    for c in columns
)
null_df = query(f'SELECT {null_exprs} FROM leads')
null_pct = null_df.T.rename(columns={0: 'null_pct'}).sort_values('null_pct', ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
null_pct['null_pct'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Null %')
ax.set_title('NULL percentage per column')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

display(null_pct)

## 3. Numeric Distributions

In [ ]:
numeric_df = query("""
    SELECT total_visits, total_time_spent, page_views_per_visit
    FROM leads
    WHERE total_visits IS NOT NULL
       OR total_time_spent IS NOT NULL
       OR page_views_per_visit IS NOT NULL
""")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cols = ['total_visits', 'total_time_spent', 'page_views_per_visit']
titles = ['Total Visits', 'Total Time Spent (min)', 'Page Views per Visit']

for ax, col, title in zip(axes, cols, titles):
    data = numeric_df[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('Numeric Feature Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Categorical Breakdowns (Top 10)

In [ ]:
cat_cols = [
    'lead_origin', 'lead_source', 'country',
    'current_occupation', 'specialization', 'last_activity'
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    df = query(f"""
        SELECT {col} AS value, COUNT(*) AS count
        FROM leads
        WHERE {col} IS NOT NULL
        GROUP BY {col}
        ORDER BY count DESC
        LIMIT 10
    """)
    ax.barh(df['value'].astype(str)[::-1], df['count'][::-1], color='steelblue')
    ax.set_title(f'Top 10: {col}')
    ax.set_xlabel('Count')

plt.suptitle('Categorical Feature Breakdowns', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Conversion Analysis

In [ ]:
overall = query("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN converted THEN 1 ELSE 0 END) AS converted_count,
        ROUND(100.0 * SUM(CASE WHEN converted THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct
    FROM leads
""")
print(f"Overall conversion rate: {overall['conversion_rate_pct'].iloc[0]}%")
display(overall)

In [ ]:
conv_by_source = query("""
    SELECT
        lead_source,
        ROUND(100.0 * SUM(CASE WHEN converted THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct,
        COUNT(*) AS total
    FROM leads
    WHERE lead_source IS NOT NULL
    GROUP BY lead_source
    ORDER BY conversion_rate_pct DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(conv_by_source['lead_source'][::-1], conv_by_source['conversion_rate_pct'][::-1], color='darkorange')
ax.set_xlabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Lead Source')
plt.tight_layout()
plt.show()

In [ ]:
conv_by_activity = query("""
    SELECT
        last_activity,
        ROUND(100.0 * SUM(CASE WHEN converted THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct,
        COUNT(*) AS total
    FROM leads
    WHERE last_activity IS NOT NULL
    GROUP BY last_activity
    ORDER BY conversion_rate_pct DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(conv_by_activity['last_activity'][::-1], conv_by_activity['conversion_rate_pct'][::-1], color='darkorange')
ax.set_xlabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Last Activity')
plt.tight_layout()
plt.show()

## 6. Correlation Matrix

In [ ]:
corr_df = query("""
    SELECT
        total_visits,
        total_time_spent,
        page_views_per_visit,
        CASE WHEN converted THEN 1 ELSE 0 END AS converted
    FROM leads
""")

corr = corr_df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    ax=ax,
    square=True,
    linewidths=0.5
)
ax.set_title('Correlation Matrix: Numeric Features vs Conversion')
plt.tight_layout()
plt.show()

display(corr)